In [ ]:
import os
import time
import copy
import json
import random
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
def find_project_root():
    current = Path.cwd().resolve()

    while current != current.parent:
        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()

DATA_ROOT = PROJECT_ROOT / "data" / "gtsrb"
SPLIT_DIR = PROJECT_ROOT / "splits"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)

In [ ]:
if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [ ]:
from Common.dataset import create_dataloaders

from Common.models import (
    ModelB,
    count_parameters
)

from Common.training import train_model

In [ ]:
train_df = pd.read_csv(
    SPLIT_DIR / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR / "test_split.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    data_root=DATA_ROOT,
    batch_size=32,
    num_workers=0
)

## Model B: Depthwise-Separable CNN

Model B keeps Model A's four-block topology and `3 -> 16 -> 32 -> 32 -> 32` channel progression. Each `3 x 3` convolution is replaced by a depthwise `3 x 3` convolution followed by a pointwise `1 x 1` convolution, then ReLU and `2 x 2` max pooling. The resulting `32 x 4 x 4` feature map feeds the same `Linear(512, 43)` classifier as Model A. ReLU avoids exponential activations, and depthwise filtering reduces multiply-accumulate work while keeping the comparison controlled.

At `64 x 64` input, Model A has **45,643** trainable parameters and about **9,459,200 MACs** per image. Model B has **25,526** trainable parameters and about **1,420,800 MACs** per image (approximately **85% fewer MACs**). Model B is below the 100,000 parameter limit.


In [ ]:
model_b = ModelB().to(device)

print(model_b)

parameters_b = count_parameters(
    model_b
)

print(
    "Model B trainable parameters:",
    parameters_b
)

Forward-pass smoke check

In [ ]:
images, labels = next(iter(train_loader))
model_b.eval()
with torch.inference_mode():
    logits = model_b(images.to(device))
assert logits.shape == (images.shape[0], 43)
assert torch.isfinite(logits).all()
print(f"Smoke check passed: {tuple(images.shape)} -> {tuple(logits.shape)}")


In [ ]:
from Common.training import (
    train_model,
    create_optimizer
)

from Common.evaluation import (
    get_predictions,
    calculate_metrics,
    plot_training_curves,
    plot_confusion_matrix,
    print_classification_report,
    total_parameters,
    estimated_fp32_size_kb,
    save_model_and_get_size,
    measure_inference_time
)

In [ ]:
selected_optimizer = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "selected_optimizer.txt"
).read_text().strip()

print(
    "Final optimizer:",
    selected_optimizer
)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


model_b = ModelB().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer_b = create_optimizer(
    selected_optimizer,
    model_b.parameters()
)

In [ ]:
model_b, history_b = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_b,
    criterion=criterion,
    epochs=30,
    device=device
)

(PROJECT_ROOT / "results" / "metrics").mkdir(parents=True, exist_ok=True)
with (PROJECT_ROOT / "results" / "metrics" / "model_b_history.json").open("w", encoding="utf-8") as history_file:
    json.dump(history_b, history_file, indent=2)


In [ ]:
FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

plot_training_curves(
    history=history_b,
    model_name="Model B",
    save_dir=FIGURES_DIR
)

In [ ]:
y_true_b, y_pred_b = get_predictions(
    model_b,
    test_loader,
    device
)

metrics_b = calculate_metrics(
    y_true_b,
    y_pred_b
)

print(metrics_b)

In [ ]:
plot_confusion_matrix(
    y_true_b,
    y_pred_b,
    model_name="Model B",
    save_dir=FIGURES_DIR
)

In [ ]:
report_b = classification_report(
    y_true_b,
    y_pred_b,
    output_dict=True,
    zero_division=0
)
print_classification_report(y_true_b, y_pred_b)

metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
with (metrics_dir / "model_b_classification_report.json").open("w", encoding="utf-8") as report_file:
    json.dump(report_b, report_file, indent=2)
np.savetxt(
    metrics_dir / "model_b_confusion_matrix.csv",
    confusion_matrix(y_true_b, y_pred_b, labels=list(range(43))),
    fmt="%d",
    delimiter=","
)


In [ ]:
average_epoch_time_b = np.mean(
    history_b["epoch_time"]
)

print(
    "Average epoch time:",
    average_epoch_time_b
)

In [ ]:
parameters_b = total_parameters(
    model_b
)

estimated_size_b = (
    estimated_fp32_size_kb(
        model_b
    )
)

model_b_path = (
    PROJECT_ROOT
    / "checkpoints"
    / "model_b_final.pth"
)

actual_size_b = (
    save_model_and_get_size(
        model_b,
        model_b_path
    )
)

print(
    "Parameters:",
    parameters_b
)

print(
    "Estimated size:",
    estimated_size_b,
    "KB"
)

print(
    "Actual size:",
    actual_size_b,
    "KB"
)

In [ ]:
inference_time_b = (
    measure_inference_time(
        model_b,
        test_loader,
        device
    )
)

print(
    "Inference time:",
    inference_time_b,
    "ms/image"
)

In [ ]:
(PROJECT_ROOT / "results" / "metrics").mkdir(parents=True, exist_ok=True)

model_b_results = {
    "Model": "Model B",
    "Parameters": parameters_b,
    "Estimated FP32 Size (KB)": estimated_size_b,
    "Actual Model Size (KB)": actual_size_b,
    "Average Epoch Time (s)": average_epoch_time_b,
    "Inference Time (ms/image)": inference_time_b,
    "Test Accuracy": metrics_b["Test Accuracy"],
    "Macro Precision": metrics_b["Macro Precision"],
    "Macro Recall": metrics_b["Macro Recall"]
}

model_b_df = pd.DataFrame(
    [model_b_results]
)

model_b_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "model_b_metrics.csv",
    index=False
)

model_b_df